**APT3025B: APPLIED MACHINE LEARNING**

**Group 9**

**NAME:** MANTAN DELA     **ID**:669212

**NAME**: RAZIYAH MWIHAKI  **ID**:670372


**Aim:** The goal of this exercise is to predict COVID-19 test results based on patient clinical data.

The initial dataset contained 45 columns, including many administrative and sparse imaging features.

In [ ]:
#Importing the dataset
import pandas as pd
import numpy as np
df = pd.read_csv(r"/coronavirusdataset.csv")
df.columns

Index(['batch_date', 'test_name', 'swab_type', 'covid19_test_results', 'age',
       'high_risk_exposure_occupation', 'high_risk_interactions', 'diabetes',
       'chd', 'htn', 'cancer', 'asthma', 'copd', 'autoimmune_dis', 'smoker',
       'temperature', 'pulse', 'sys', 'dia', 'rr', 'sats', 'rapid_flu_results',
       'rapid_strep_results', 'ctab', 'labored_respiration', 'rhonchi',
       'wheezes', 'days_since_symptom_onset', 'cough', 'cough_severity',
       'fever', 'sob', 'sob_severity', 'diarrhea', 'fatigue', 'headache',
       'loss_of_smell', 'loss_of_taste', 'runny_nose', 'muscle_sore',
       'sore_throat', 'cxr_findings', 'cxr_impression', 'cxr_label',
       'cxr_link'],
      dtype='object')

**Data Preprocessing & Cleaning**

We transformed the raw dataset into a refined input matrix by following these specific steps:
-  We discarded 15 irrelevant columns, including administrative markers (batch_date, test_name) and sparse medical imaging data (cxr_link, cxr_findings) to eliminated noise and prevented the model from establishing false correlations.
-  We addressed missing values by applying a median imputation strategy for numerical features to ensure that extreme clinical outliers do not skew the feature distributions.
- For missing values in Categorical features, We preserved data by filling null entries with the label 'Unknown'. This allows the model to interpret missing data as a distinct category rather than simply discarding the observation.


In [ ]:
from sklearn.impute import SimpleImputer
# Data cleaning
#Drop Administrative columns or Columns missing more than 80% of the data
cols_to_drop = ['batch_date', 'test_name', 'swab_type', 'cxr_link', 'cxr_label', 'cxr_findings', 'cxr_impression', 'rapid_flu_results', 'rapid_strep_results', 'cough_severity', 'sob_severity', 'days_since_symptom_onset', 'rhonchi', 'wheezes', 'ctab']
df_cleaned = df.drop(columns=cols_to_drop)
print("Remaining columns:", df_cleaned.columns.tolist())

#Handling missing values in Numerical columns
numeric_cols = ['age', 'temperature', 'pulse', 'sys', 'dia', 'rr', 'sats']
imputer = SimpleImputer(strategy='median')
df_cleaned[numeric_cols] = imputer.fit_transform(df_cleaned[numeric_cols])

#Handling missing values in Categorical columns
categorical_cols = ['high_risk_exposure_occupation', 'high_risk_interactions', 'diabetes',
                    'chd', 'htn', 'cancer', 'asthma', 'copd', 'autoimmune_dis', 'smoker',
                    'labored_respiration', 'cough', 'fever', 'sob', 'diarrhea', 'fatigue',
                    'headache', 'loss_of_smell', 'loss_of_taste', 'runny_nose', 'muscle_sore', 'sore_throat']

df_cleaned[categorical_cols] = df_cleaned[categorical_cols].fillna('Unknown')
#Confirm whether there are still some missing values
print("Total missing values remaining:", df_cleaned.isnull().sum().sum())

Remaining columns: ['covid19_test_results', 'age', 'high_risk_exposure_occupation', 'high_risk_interactions', 'diabetes', 'chd', 'htn', 'cancer', 'asthma', 'copd', 'autoimmune_dis', 'smoker', 'temperature', 'pulse', 'sys', 'dia', 'rr', 'sats', 'labored_respiration', 'cough', 'fever', 'sob', 'diarrhea', 'fatigue', 'headache', 'loss_of_smell', 'loss_of_taste', 'runny_nose', 'muscle_sore', 'sore_throat']
Total missing values remaining: 0


**Encoding**
- Target Variable: We applied a LabelEncoder to the COVID-19 test results to convert them into binary format ($0$ for Negative, $1$ for Positive).
- Feature Set: We utilized pd.get_dummies with drop_first=True to transform categorical symptoms into a numerical format compatible with the SVC while avoiding the "dummy variable trap."

In [ ]:
#Encoding categorical Columns
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_cleaned['covid19_test_results'] = le.fit_transform(df_cleaned['covid19_test_results'])
df_final = pd.get_dummies(df_cleaned, columns=categorical_cols, drop_first=True)

### **Training and Scaling Comparison**
- Baseline Model: We trained an initial SVC using raw, unscaled data.- Normalized Model: We adjusted C and gamma values and implemented a MinMaxScaler to squeeze all features into a uniform range to ensure that features with larger numerical ranges, such as pulse, do not overwhelm binary features during the SVC's hyperplane calculation.

In [ ]:
#Training the model
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

X = df_final.drop(columns=['covid19_test_results'])
y = df_final['covid19_test_results']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

model = SVC(kernel="rbf", C=1, gamma=0.03).fit(X_train, y_train)
predictions = model.predict(X_test)

print("Training Score: ", model.score(X_train, y_train))
print("Test Score: ", model.score(X_test, y_test))
print("First 10 Predictions without Scaling", predictions[:10])
print("First 10 Actual Results:", y_test.values[:10])

Training Score:  0.9978062157221207
Test Score:  0.9939692982456141
First 10 Predictions without Scaling [0 0 0 0 0 0 0 0 0 0]
First 10 Actual Results: [0 0 0 0 0 0 0 0 0 0]


In [ ]:
#Training with scaled data
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

X = df_final.drop(columns=['covid19_test_results'])
y = df_final['covid19_test_results']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

scaler = MinMaxScaler()
X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)

model = SVC(kernel="rbf", C=1000, gamma=0.001).fit(X_train_scale, y_train)
predictions = model.predict(X_test_scale)

print("Training Score: ", model.score(X_train_scale, y_train))
print("Test Score: ", model.score(X_test_scale, y_test))
print("First 10 Predictions after Scaling", predictions[:10])
print("First 10 Actual Results:", y_test.values[:10])

Training Score:  0.9974405850091408
Test Score:  0.9939692982456141
First 10 Predictions after Scaling [0 0 0 0 0 0 0 0 0 0]
First 10 Actual Results: [0 0 0 0 0 0 0 0 0 0]


**Test Results and Observations**

The model achieved the following performance metrics
MetricUnscaled : Training Score 99.78%, Test Score 99.40%
ModelScaled : Training Score 99.74%, Test Score99.40%
- While the accuracy scores exceed 99%, the model failed to identify a single positive case, resulting in a 0% Recall for the positive class.

- In this case, the Training Score (99.74%) and the Test Score (99.40%) are nearly identical. The model is generalizing. However, it's strategy is to ignore the minority class (Positive) because it is statistically insignificant to the overall accuracy. The model has not failed to generalize; it has failed to recognize the importance of the rare "Positive" cases due to their low frequency in the data.

**Conclusion**

The current model effectively demonstrates a clean data-processing pipeline and the technical implementation of SVC scaling. However, the extreme disparity between classes renders the model clinically ineffective for screening.
Shifting the focus from raw accuracy to a Confusion Matrix will provide critical insight into Recall and False Negatives, which are vital for medical safety.
Also, To improve performance, transitioning to a Random Forest Classifier which is superior as it inherently supports balanced class weights to handle rare positive cases and provide feature importance to identify key symptoms will improve prediction and clinical utility of the model.